In [ ]:
# Full refactored version of CNN pipeline using fixed hyperparameters
# and 80/20 train-test split (no Optuna), with visualization support

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import matplotlib.pyplot as plt

if "__file__" in globals():
    # Running as a script
    project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
else:
    # Running in Jupyter
    project_root = os.path.abspath("..")

sys.path.append(project_root)

from ReusableFunctions.DataPreprocessing import DataPreprocessing
from ReusableFunctions.EvaluationMetrics import EvaluationMetrics as EM
from reproducibility_settings import set_global_seed

# Reproducibility
set_global_seed(seed=42, framework='torch')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


class CNNModel(nn.Module):
    def __init__(self, input_shape, filters, kernel_size, pooling_size, dropout_rate, activation):
        super(CNNModel, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(in_channels=input_shape[0], out_channels=filters, kernel_size=kernel_size, padding=padding)
        self.pool = nn.MaxPool1d(kernel_size=pooling_size)
        self.dropout = nn.Dropout(dropout_rate)

        if activation == "relu":
            self.activation = nn.ReLU()
        elif activation == "leaky_relu":
            self.activation = nn.LeakyReLU(negative_slope=0.01)
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        with torch.no_grad():
            dummy_input = torch.zeros(1, input_shape[0], input_shape[1])
            x = self.pool(self.activation(self.conv1(dummy_input)))
            flattened_size = x.view(1, -1).shape[1]

        self.fc1 = nn.Linear(flattened_size, 1)

    def forward(self, x):
        x = self.activation(self.conv1(x))
        x = self.pool(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x


def train_and_evaluate_model(model, train_loader, test_data, optimizer, criterion, scaler_y, forecast_window, epochs):
    X_test_tensor = torch.tensor(test_data[0], dtype=torch.float32).permute(0, 2, 1).to(device)
    y_test_scaled_tensor = torch.tensor(test_data[1], dtype=torch.float32).to(device)

    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(), yb.squeeze())
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_preds_scaled = model(X_test_tensor).squeeze().cpu().numpy()
        test_preds_unscaled = scaler_y.inverse_transform(test_preds_scaled.reshape(-1, 1))
        y_true_unscaled = scaler_y.inverse_transform(test_data[1])

    r2 = EM.r2(y_true_unscaled, test_preds_unscaled)
    if np.isnan(r2) or np.isinf(r2):
        rmse, mape, acc = np.nan, np.nan, np.nan
    else:
        rmse = EM.rmse(y_true_unscaled, test_preds_unscaled)
        mape = EM.mape(y_true_unscaled, test_preds_unscaled)
        acc = EM.accuracy(y_true_unscaled, test_preds_unscaled)

    profit_index = EM.profitability_index(y_true_unscaled, test_preds_unscaled, forecast_window)

    # Print metrics
    print("\n=== Evaluation Metrics ===")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"R2: {r2:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Profitability Index: {profit_index:.4f}")

    # Plot results
    plt.figure(figsize=(12, 5))
    plt.plot(y_true_unscaled, label='Actual')
    plt.plot(test_preds_unscaled, label='Predicted')
    plt.title("Predicted vs Actual Prices on Test Set")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return rmse, mape, r2, acc, profit_index


def split_dataset(X, y, train_ratio=0.8):
    split_index = int(len(X) * train_ratio)
    return X[:split_index], X[split_index:], y[:split_index], y[split_index:]


if __name__ == '__main__':
    ticker = 'QCOM'
    os.makedirs('check', exist_ok=True)

    # Fixed technical indicator combination
    fixed_indicator_combo = ['20MA', '50MA','MACD', 'Upper_BB', 'CCI', 'ATR']

    window_size, forecast_window = 60, 1

    data_processor = DataPreprocessing(ticker=ticker, start_date='2014-01-01', end_date='2024-12-31')
    df = data_processor.add_technical_indicators()

    selected_features = ['Close'] + fixed_indicator_combo
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df[selected_features])
    X, y = data_processor.create_windowed_data(scaled_data, window_size, forecast_window)
    X_train, X_test, y_train, y_test = split_dataset(X, y, train_ratio=0.8)

    scaler_y = MinMaxScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

    fixed_params = {
        "filters": 128,
        "kernel_size": 3,
        "pooling_size": 2,
        "dropout": 0.3,
        "lr": 0.0005,
        "batch_size": 64,
        "activation": "relu",
        "epochs": 150
    }

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32).permute(0, 2, 1),
                      torch.tensor(y_train_scaled, dtype=torch.float32)),
        batch_size=fixed_params["batch_size"], shuffle=False
    )

    model = CNNModel(
        input_shape=(X_train.shape[2], X_train.shape[1]),
        filters=fixed_params["filters"],
        kernel_size=fixed_params["kernel_size"],
        pooling_size=fixed_params["pooling_size"],
        dropout_rate=fixed_params["dropout"],
        activation=fixed_params["activation"]
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=fixed_params["lr"])
    criterion = nn.MSELoss()

    print(f"\nRunning for Fixed Indicators: {fixed_indicator_combo}, Window Size: {window_size}, Forecast: {forecast_window}")
    train_and_evaluate_model(
        model, train_loader, (X_test, y_test_scaled), optimizer, criterion,
        scaler_y, forecast_window, fixed_params["epochs"]
    )
